In [0]:
# Databricks environment configuration
catalog_name = "learning_projects"
schema_name = "retail_project"

table_prefix = f"{catalog_name}.{schema_name}"

In [0]:
silver_orders_df = spark.table(f"{table_prefix}.silver_orders")
silver_order_items_df = spark.table(f"{table_prefix}.silver_order_items")
dim_customer_df = spark.table(f"{table_prefix}.dim_customer")
dim_product_df = spark.table(f"{table_prefix}.dim_product")

print("silver_orders: ", silver_orders_df.count())
print("silver_order_items: ", silver_order_items_df.count())
print("dim_customer: ", dim_customer_df.count())
print("dim_product: ", dim_product_df.count())

In [0]:
from pyspark.sql.functions import col
from pyspark.sql.types import DecimalType

money_type = DecimalType(18, 2)

sales_base_df = (
    silver_order_items_df.alias("soi")
    .join(
        silver_orders_df.alias("so"),
        col("soi.order_id") == col("so.order_id"),
        "inner"
    )
    .select(
        col("so.order_id").alias("order_id"),
        col("soi.order_item_id").alias("order_item_id"),
        col("so.customer_id").alias("customer_id"),
        col("soi.product_id").alias("product_id"),
        col("soi.seller_id").alias("seller_id"),
        col("so.order_purchase_timestamp").alias("order_purchase_timestamp"),
        col("so.order_status").alias("order_status"),
        col("soi.price").alias("price"),
        col("soi.freight_value").alias("freight_value"),
        col("so.is_late_delivery").alias("is_late_delivery")
    )
    .withColumn("price", col("price").cast(money_type))
    .withColumn("freight_value", col("freight_value").cast(money_type))
    .withColumn(
        "line_total",
        col("price") + col("freight_value")
    )
)

In [0]:
from pyspark.sql.functions import sum as spark_sum

sales_base_df.agg(
    spark_sum("price").alias("total_product_sales"),
    spark_sum("freight_value").alias("total_freight"),
    spark_sum("line_total").alias("total_revenue")
).show()

In [0]:
duplicate_fact_grain_df = (
    sales_base_df
    .groupBy("order_id", "order_item_id")
    .count()
    .filter(col("count") > 1)
)

print(
    "Duplicate fact grain:",
    duplicate_fact_grain_df.count()
)

In [0]:
silver_customers_df = spark.table(
    f"{table_prefix}.silver_customers"
)


In [0]:
sales_customer_df = (
    sales_base_df.alias("s")
    .join(
        silver_customers_df.alias("c"),
        col("s.customer_id") == col("c.customer_id"),
        "left"
    )
    .select(
        "s.*",
        col("c.customer_unique_id").alias("customer_unique_id")
    )
)

print("Rows after customer mapping:", sales_customer_df.count())

print(
    "Missing customer_unique_id:",
    sales_customer_df
    .filter(col("customer_unique_id").isNull())
    .count()
)

In [0]:
from pyspark.sql.functions import col

sales_customer_sk_df = (
    sales_customer_df.alias("s").join(
        dim_customer_df.alias("d"), 
        (col("s.customer_unique_id") == col("d.customer_unique_id")) &
        (col("s.order_purchase_timestamp") >= col("d.effective_from")) &
        ((col("s.order_purchase_timestamp") < col("d.effective_to"))
         | (col("d.effective_to").isNull())), "left" 
    )
    .select(
        "s.*",
        col("d.customer_sk").alias("customer_sk")
    )
)
print("Rows after SCD2 join:", sales_customer_sk_df.count())

print(
    "Missing customer_sk:",
    sales_customer_sk_df
    .filter(col("customer_sk").isNull())
    .count()
)

In [0]:
from pyspark.sql.functions import col

base_count = sales_customer_df.count()
scd2_count = sales_customer_sk_df.count()

missing_customer_sk = (
    sales_customer_sk_df
    .filter(col("customer_sk").isNull())
    .count()
)

duplicate_fact_grain = (
    sales_customer_sk_df
    .groupBy("order_id", "order_item_id")
    .count()
    .filter(col("count") > 1)
    .count()
)

print("Rows before SCD2 join:", base_count)
print("Rows after SCD2 join:", scd2_count)
print("Missing customer surrogate keys:", missing_customer_sk)
print("Duplicate fact grain:", duplicate_fact_grain)
print("Row counts match:", base_count == scd2_count)

In [0]:
sales_product_sk_df = (
    sales_customer_sk_df.alias("s").join(
        dim_product_df.alias("p"), (col("s.product_id") == col("p.product_id")), "left")
    .select(
        "s.*",
        col("p.product_sk").alias("product_sk")
    )
)
print("Rows after product join:", sales_product_sk_df.count())

print(
    "Missing product_sk:",
    sales_product_sk_df
    .filter(col("product_sk").isNull())
    .count()
)

In [0]:
from pyspark.sql.functions import col

before_product_join = sales_customer_sk_df.count()
after_product_join = sales_product_sk_df.count()

missing_product_sk = (
    sales_product_sk_df
    .filter(col("product_sk").isNull())
    .count()
)

duplicate_fact_grain = (
    sales_product_sk_df
    .groupBy("order_id", "order_item_id")
    .count()
    .filter(col("count") > 1)
    .count()
)

print("Rows before product join:", before_product_join)
print("Rows after product join:", after_product_join)
print("Missing product surrogate keys:", missing_product_sk)
print("Duplicate fact grain:", duplicate_fact_grain)
print("Row counts match:", before_product_join == after_product_join)

In [0]:
sales_product_sk_df.show(10)

In [0]:
from pyspark.sql.types import DecimalType
from pyspark.sql.functions import col

money_type = DecimalType(18,2)

fact_sales_df = (
    sales_product_sk_df
    .select(
        "order_id",
        "order_item_id",
        "customer_sk",
        "product_sk",
        "seller_id",
        "order_purchase_timestamp",
        "order_status",
        "price",
        "freight_value",
        "line_total",
        "is_late_delivery"
    )
    .withColumn("price", col("price").cast(money_type))
    .withColumn("freight_value", col("freight_value").cast(money_type))
    .withColumn("line_total", col("line_total").cast(money_type))

)
print("Fact rows:", fact_sales_df.count())
duplicate_grain_df = (
    fact_sales_df
    .groupBy("order_id", "order_item_id")
    .count()
    .filter(col("count") > 1)
)

print("Duplicate fact grain:", duplicate_grain_df.count())

fact_sales_df.printSchema()


In [0]:
from pyspark.sql.functions import col

print("Total fact rows:", fact_sales_df.count())

print(
    "Null customer keys:",
    fact_sales_df.filter(col("customer_sk").isNull()).count()
)

print(
    "Null product keys:",
    fact_sales_df.filter(col("product_sk").isNull()).count()
)

print(
    "Null monetary values:",
    fact_sales_df.filter(
        col("price").isNull() |
        col("freight_value").isNull() |
        col("line_total").isNull()
    ).count()
)

print(
    "Incorrect line totals:",
    fact_sales_df.filter(
        col("line_total") !=
        (col("price") + col("freight_value"))
    ).count()
)

In [0]:
from pyspark.sql.functions import sum
(
    fact_sales_df
    .agg(
        sum("price").alias("total_product_sales"),
        sum("freight_value").alias("total_freight"),
        sum("line_total").alias("total_revenue")
    )
).show()

In [0]:
(
    fact_sales_df
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(f"{table_prefix}.fact_sales")
)

In [0]:
fact_validation_df = spark.sql(f"""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(DISTINCT order_id, order_item_id) AS distinct_fact_grain,
        SUM(price) AS product_sales,
        SUM(freight_value) AS freight,
        SUM(line_total) AS revenue
    FROM {table_prefix}.fact_sales
""")

fact_validation_df.show()

# %sql
# SELECT
#     COUNT(*) AS total_rows,
#     COUNT(DISTINCT CONCAT(order_id, '|', order_item_id)) AS distinct_fact_grain,
#     SUM(price) AS product_sales,
#     SUM(freight_value) AS freight,
#     SUM(line_total) AS revenue
# FROM learning_projects.retail_project.fact_sales;
# replace this query